# End-to-End ETL & Data Analysis Pipeline
## Domain: Student Academic Performance & Learning Analytics
**Student:** Prajjwal Gyawali  
**Dataset Source:** [Kaggle Student Dropout & Performance Dataset](https://www.kaggle.com/datasets/muhammadkhubaibahmad/student-performance-and-clustering-dataset)  
**Repository:** [student-performance-analysis](https://github.com/PrajjwalGyawali/student-performance-analysis.git)  

---
### Project Architecture Lifecycle
```
Raw Data → Extract → EDA & Quality Checks → Clean & Transform → Normalize (4 Entities) → SQLite DB → SQL Reporting
```

### 1. Library Imports & Environment Configuration

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

sns.set_theme(style='whitegrid', palette='muted')
print('Dependencies loaded successfully!')

### Phase 1: Data Acquisition & Extraction
Preserving raw data and loading into Pandas DataFrame.

In [ ]:
raw_path = os.path.join('..', 'data', 'raw', 'student_dropout_behavior_raw.csv')
df_raw = pd.read_csv(raw_path)
print(f'Raw Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
df_raw.head()

### Phase 2: Exploratory Data Analysis & Data Quality Audit
Checking for null values, duplicates, outliers, and distribution shapes.

In [ ]:
print('Missing Values:\n', df_raw.isnull().sum())
print('\nDuplicate Records:', df_raw.duplicated().sum())
df_raw.describe().round(2)

### Phase 3: Data Cleaning, Feature Engineering & Normalization
- Impute missing `assignments_submitted`.
- Standardize string formatting.
- Calculate attendance percentages, composite scores, letter grades, and dropout risk.
- Normalize into 4 relational DataFrames: `students`, `courses`, `attendance_records`, `academic_assessments`.

In [ ]:
import sys
sys.path.insert(0, os.path.join('..', 'src'))
from transform import clean_and_transform_data

tables = clean_and_transform_data(df_raw)
print('Entity tables normalized:')
for name, tbl in tables.items():
    if isinstance(tbl, pd.DataFrame):
        print(f'  - {name}: {tbl.shape}')

### Phase 4: Data Analysis & Business Insights
Evaluating 5 core analytical questions and deriving data-backed insights.

In [ ]:
from analysis import perform_data_analysis
analysis_output = perform_data_analysis(tables, save_plots=False)
print('\nKey Findings:')
for insight in analysis_output['insights']:
    print('-', insight)

### Phase 5 & 6: Database Loading (SQLAlchemy + SQLite)
Loading normalized DataFrames into `database/student_performance.db` with primary and foreign key constraints.

In [ ]:
from load import load_data_to_database, DB_PATH
counts = load_data_to_database(tables, db_path=DB_PATH)
print('Loaded record counts in SQLite database:', counts)

### Phase 7: SQL Analytical Reporting
Running multi-table queries with JOINs, Aggregates, CASE expressions, Subqueries, and Window Functions.

In [ ]:
from sql_reporting import generate_sql_reports
reports = generate_sql_reports(db_path=DB_PATH)
print(f'Successfully ran {len(reports)} SQL analytical reports!')